# Sink Node Characterisation — Which Nodes Become Sinks?

**Hypothesis H2:** Attention sinks in graph transformers form on nodes with specific
structural properties (high degree, high centrality, spectral extremity).

This notebook runs inference on trained models to extract **per-node** sink scores
and correlates them with graph-theoretic node properties. We analyse PascalVOC
(strongest sinks, ~479 nodes) and ZINC (smallest graphs, ~23 nodes) for contrast.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy.stats import spearmanr
import yaml
import networkx as nx
from tqdm import tqdm
from torch_geometric.utils import degree, get_laplacian, to_scipy_sparse_matrix

from src.model import InstrumentedGPS
from src.datasets import get_dataloaders, DATASET_INFO
from src.metrics import compute_sink_scores

matplotlib.rcParams.update({
    'font.size': 11,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

OUTPUTS_DIR = '../outputs'
FIGURES_DIR = '../outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def load_experiment(experiment_id, device):
    """Load a trained model and its config."""
    config_path = os.path.join(OUTPUTS_DIR, experiment_id, 'config.yaml')
    with open(config_path) as f:
        config = yaml.safe_load(f)
    dataset_info = DATASET_INFO[config['data']['dataset']].copy()
    if config['vnode']['enabled'] and dataset_info.get('num_node_types') is not None:
        dataset_info['num_node_types'] = config['vnode']['num_node_types']
        dataset_info['num_edge_types'] = config['vnode']['num_edge_types']
    model = InstrumentedGPS(config, dataset_info).to(device)
    model.load_state_dict(torch.load(
        os.path.join(OUTPUTS_DIR, experiment_id, 'best_model.pt'),
        map_location=device, weights_only=True
    ))
    model.eval()
    return model, config, dataset_info

## 1. Extract per-node sink scores and structural properties

For each graph, we extract:
- Per-node sink scores at each layer
- Node degree
- Betweenness centrality
- Fiedler vector value (spectral position)

In [ ]:
@torch.no_grad()
def extract_sink_data(model, config, device, max_graphs=200):
    """Extract per-node sink scores and structural properties for each graph."""
    _, _, test_loader, _ = get_dataloaders(config)
    model.eval()
    model._register_attn_hooks()
    
    all_graphs = []
    graphs_done = 0
    
    for batch in tqdm(test_loader, desc='Extracting sink data'):
        if graphs_done >= max_graphs:
            break
        batch = batch.to(device)
        _ = model(batch, collect_diagnostics=True)
        
        batch_ids = model.layer_data[0]['batch']
        unique_graphs = batch_ids.unique()
        
        for g_idx, g_id in enumerate(unique_graphs):
            if graphs_done >= max_graphs:
                break
            
            graph_mask = (batch_ids == g_id)
            num_nodes_g = graph_mask.sum().item()
            
            # Build local edge index
            node_indices = torch.where(graph_mask)[0]
            g_edges = batch.edge_index[:, batch.batch[batch.edge_index[0]] == g_id]
            local_map = {int(n): i for i, n in enumerate(node_indices)}
            local_edges = []
            for s, d in g_edges.t().tolist():
                if s in local_map and d in local_map:
                    local_edges.append([local_map[s], local_map[d]])
            
            if local_edges:
                local_ei = torch.tensor(local_edges).t()
            else:
                local_ei = torch.zeros(2, 0, dtype=torch.long)
            
            # Degree
            local_src = local_ei[0] if local_ei.size(1) > 0 else torch.zeros(0, dtype=torch.long)
            deg = degree(local_src, num_nodes=num_nodes_g)
            
            # Fiedler vector
            if num_nodes_g > 2 and local_ei.size(1) > 0:
                ei_lap, ew_lap = get_laplacian(local_ei, normalization='sym', num_nodes=num_nodes_g)
                L = to_scipy_sparse_matrix(ei_lap, ew_lap, num_nodes=num_nodes_g).toarray()
                eigenvalues, eigenvectors = np.linalg.eigh(L)
                fiedler = eigenvectors[:, 1]
            else:
                fiedler = np.zeros(num_nodes_g)
            
            # Betweenness centrality
            G = nx.Graph()
            G.add_nodes_from(range(num_nodes_g))
            for s, d in zip(local_ei[0].tolist(), local_ei[1].tolist()):
                G.add_edge(s, d)
            centrality = nx.betweenness_centrality(G)
            cent_array = np.array([centrality.get(i, 0.0) for i in range(num_nodes_g)])
            
            # Per-layer sink scores
            graph_data = {
                'num_nodes': num_nodes_g,
                'degree': deg.numpy(),
                'fiedler': fiedler,
                'centrality': cent_array,
                'sink_scores_by_layer': {},
                'sink_rate_by_layer': {},
            }
            
            for layer_idx in range(1, model.num_layers + 1):
                attn_full = model.attn_weights[layer_idx - 1]
                if attn_full is not None and g_idx < attn_full.size(0):
                    attn_g = attn_full[g_idx, :, :num_nodes_g, :num_nodes_g]
                    sink_data = compute_sink_scores(attn_g)
                    graph_data['sink_scores_by_layer'][layer_idx] = sink_data['sink_score_per_node'].numpy()
                    graph_data['sink_rate_by_layer'][layer_idx] = sink_data['sink_rate_per_node'].numpy()
            
            all_graphs.append(graph_data)
            graphs_done += 1
    
    model._remove_attn_hooks()
    print(f'Collected data for {len(all_graphs)} graphs')
    return all_graphs

In [ ]:
# Extract for PascalVOC (no VNode) and ZINC (no VNode)
# These show natural sink formation without the VNode artefact

experiments_to_analyse = [
    ('pascal-novnode', 'PascalVOC (no VNode, ~479 nodes)'),
    ('zinc-novnode', 'ZINC (no VNode, ~23 nodes)'),
]

all_sink_data = {}
for eid, desc in experiments_to_analyse:
    print(f'\n=== {desc} ===')
    model, config, _ = load_experiment(eid, device)
    all_sink_data[eid] = {
        'graphs': extract_sink_data(model, config, device, max_graphs=200),
        'num_layers': model.num_layers,
        'desc': desc,
    }
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 2. Sink Score vs Structural Properties (last layer)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, (eid, data) in enumerate(all_sink_data.items()):
    graphs = data['graphs']
    last_layer = data['num_layers']
    desc = data['desc']
    
    degrees, sink_scores = [], []
    fiedler_vals, sink_scores_f = [], []
    cent_vals, sink_scores_c = [], []
    
    for g in graphs:
        if last_layer not in g['sink_scores_by_layer']:
            continue
        scores = g['sink_scores_by_layer'][last_layer]
        degrees.extend(g['degree'].tolist())
        sink_scores.extend(scores.tolist())
        fiedler_vals.extend(np.abs(g['fiedler']).tolist())
        sink_scores_f.extend(scores.tolist())
        cent_vals.extend(g['centrality'].tolist())
        sink_scores_c.extend(scores.tolist())
    
    degrees = np.array(degrees)
    sink_scores_arr = np.array(sink_scores)
    fiedler_vals = np.array(fiedler_vals)
    sink_scores_f = np.array(sink_scores_f)
    cent_vals = np.array(cent_vals)
    sink_scores_c = np.array(sink_scores_c)
    
    # Degree
    r_deg, p_deg = spearmanr(degrees, sink_scores_arr)
    axes[row, 0].scatter(degrees, sink_scores_arr, alpha=0.1, s=6, color='tab:red')
    axes[row, 0].set_xlabel('Node Degree')
    axes[row, 0].set_ylabel('Sink Score')
    axes[row, 0].set_title(f'{desc}\nDegree (r={r_deg:.3f}, p={p_deg:.1e})')
    axes[row, 0].grid(True, alpha=0.3)
    
    # Fiedler
    r_fied, p_fied = spearmanr(fiedler_vals, sink_scores_f)
    axes[row, 1].scatter(fiedler_vals, sink_scores_f, alpha=0.1, s=6, color='tab:blue')
    axes[row, 1].set_xlabel('|Fiedler Vector Value|')
    axes[row, 1].set_ylabel('Sink Score')
    axes[row, 1].set_title(f'Spectral Extremity (r={r_fied:.3f}, p={p_fied:.1e})')
    axes[row, 1].grid(True, alpha=0.3)
    
    # Centrality
    r_cent, p_cent = spearmanr(cent_vals, sink_scores_c)
    axes[row, 2].scatter(cent_vals, sink_scores_c, alpha=0.1, s=6, color='tab:green')
    axes[row, 2].set_xlabel('Betweenness Centrality')
    axes[row, 2].set_ylabel('Sink Score')
    axes[row, 2].set_title(f'Centrality (r={r_cent:.3f}, p={p_cent:.1e})')
    axes[row, 2].grid(True, alpha=0.3)

plt.suptitle('Sink Score vs Structural Properties (Last Layer)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig7_sink_characterisation.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 3. Is the sink node always the highest-degree node?

In [ ]:
for eid, data in all_sink_data.items():
    graphs = data['graphs']
    last_layer = data['num_layers']
    desc = data['desc']
    
    sink_is_max_deg = 0
    sink_is_top3_deg = 0
    sink_is_max_cent = 0
    total = 0
    
    for g in graphs:
        if last_layer not in g['sink_scores_by_layer']:
            continue
        scores = g['sink_scores_by_layer'][last_layer]
        sink_node = np.argmax(scores)
        deg = g['degree']
        cent = g['centrality']
        
        total += 1
        deg_rank = int((deg >= deg[sink_node]).sum())
        if deg_rank == 1:
            sink_is_max_deg += 1
        if deg_rank <= 3:
            sink_is_top3_deg += 1
        cent_rank = int((cent >= cent[sink_node]).sum())
        if cent_rank == 1:
            sink_is_max_cent += 1
    
    print(f'\n=== {desc} ({total} graphs) ===')
    print(f'  Sink = max-degree node:      {sink_is_max_deg:>4}/{total} ({100*sink_is_max_deg/total:.1f}%)')
    print(f'  Sink = top-3 degree node:    {sink_is_top3_deg:>4}/{total} ({100*sink_is_top3_deg/total:.1f}%)')
    print(f'  Sink = max-centrality node:  {sink_is_max_cent:>4}/{total} ({100*sink_is_max_cent/total:.1f}%)')

## 4. Sink stability across layers

Does the same node remain the sink across all layers, or does the sink identity shift?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (eid, data) in zip(axes, all_sink_data.items()):
    graphs = data['graphs']
    desc = data['desc']
    
    consistency_scores = []
    for g in graphs:
        sink_nodes = []
        for l in sorted(g['sink_scores_by_layer'].keys()):
            sink_nodes.append(np.argmax(g['sink_scores_by_layer'][l]))
        if len(sink_nodes) > 1:
            final_sink = sink_nodes[-1]
            consistency = np.mean([1.0 if s == final_sink else 0.0 for s in sink_nodes])
            consistency_scores.append(consistency)
    
    ax.hist(consistency_scores, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(np.mean(consistency_scores), color='red', linestyle='--',
               label=f'Mean: {np.mean(consistency_scores):.2f}')
    ax.set_xlabel('Consistency (fraction of layers where sink = final sink)')
    ax.set_ylabel('Number of graphs')
    ax.set_title(desc)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Sink Node Identity Stability Across Layers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig8_sink_stability.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 5. Sink score distribution across all nodes

How concentrated is the sink? Is it one dominant node or several?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (eid, data) in zip(axes, all_sink_data.items()):
    graphs = data['graphs']
    last_layer = data['num_layers']
    desc = data['desc']
    
    # For each graph, compute ratio: max_sink_score / second_max_sink_score
    dominance_ratios = []
    for g in graphs:
        if last_layer not in g['sink_scores_by_layer']:
            continue
        scores = np.sort(g['sink_scores_by_layer'][last_layer])[::-1]
        if len(scores) >= 2 and scores[1] > 1e-8:
            dominance_ratios.append(scores[0] / scores[1])
    
    ax.hist(dominance_ratios, bins=30, color='darkorange', edgecolor='white', alpha=0.8)
    ax.axvline(np.median(dominance_ratios), color='red', linestyle='--',
               label=f'Median: {np.median(dominance_ratios):.2f}')
    ax.set_xlabel('Dominance ratio (1st / 2nd sink score)')
    ax.set_ylabel('Number of graphs')
    ax.set_title(desc)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Sink Dominance: Is There One Clear Sink or Several?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig9_sink_dominance.pdf'), bbox_inches='tight', dpi=150)
plt.show()

## 6. VNode experiment: does the VNode always become the sink?

When VNode is present, check whether it absorbs the sink role.

In [ ]:
vnode_experiments = [
    ('pascal-vnode', 'PascalVOC with VNode'),
    ('zinc-vnode', 'ZINC with VNode'),
]

for eid, desc in vnode_experiments:
    print(f'\n=== {desc} ===')
    model, config, _ = load_experiment(eid, device)
    vnode_graphs = extract_sink_data(model, config, device, max_graphs=100)
    last_layer = model.num_layers
    
    vnode_is_sink = 0
    total = 0
    vnode_sink_scores = []
    
    for g in vnode_graphs:
        if last_layer not in g['sink_scores_by_layer']:
            continue
        scores = g['sink_scores_by_layer'][last_layer]
        total += 1
        # VNode is the last node
        vnode_idx = g['num_nodes'] - 1
        sink_node = np.argmax(scores)
        if sink_node == vnode_idx:
            vnode_is_sink += 1
        if vnode_idx < len(scores):
            vnode_sink_scores.append(scores[vnode_idx])
    
    print(f'  VNode is the sink node: {vnode_is_sink}/{total} ({100*vnode_is_sink/total:.1f}%)')
    print(f'  VNode avg sink score: {np.mean(vnode_sink_scores):.4f}')
    print(f'  VNode median sink score: {np.median(vnode_sink_scores):.4f}')
    
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 7. Summary

Key findings from sink node characterisation:
1. **Which nodes become sinks?** Correlations with degree, centrality, spectral position
2. **Is it consistent?** Sink stability across layers
3. **Is it dominant?** One clear sink vs distributed
4. **VNode effect:** Does VNode consistently absorb the sink role?